# Category 8 - WebSocket Architecture Comparison

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares transport architectures for real-time assistive AI. Ally Vision v2 currently uses persistent WebSockets end-to-end, so the comparison emphasizes duplex audio, interruption support, session reuse, browser fit, and implementation complexity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
# Hardcoded from public-source URLs and project-grounded measurements only.
# No runtime web calls are performed in this notebook.
data = {
  "source_urls": {
    "MDN WebSocket": "https://developer.mozilla.org/en-US/docs/Web/API/WebSocket",
    "MDN SSE": "https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events",
    "gRPC core concepts": "https://grpc.io/docs/what-is-grpc/core-concepts/",
    "LiveKit connect docs": "https://docs.livekit.io/intro/basics/connect/",
    "Project realtime client": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py"
  },
  "comparison_rows": [
    {
      "Metric": "Persistent session reuse",
      "WebSocket (Ally choice)": "Yes, current code reuses DashScope session [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
      "REST polling": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/HTTP]",
      "SSE": "Long-lived but one-way [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
      "gRPC streaming": "Yes [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
      "LiveKit / WebRTC": "Yes, room/session model [source: https://docs.livekit.io/intro/basics/connect/]",
      "MQTT": "Yes [source: https://mqtt.org/]",
      "Socket.IO": "Yes [source: https://socket.io/docs/v4/]",
      "HTTP/3 + QUIC streams": "Yes [source: https://developer.mozilla.org/en-US/docs/Glossary/QUIC]",
      "Pipecat": "N/A (not publicly available) [source: https://github.com/pipecat-ai/pipecat]",
      "Daily.co": "N/A (not publicly available) [source: https://docs.daily.co/]",
      "Agora RTE": "N/A (not publicly available) [source: https://docs.agora.io/en/real-time-engagement]"
    },
    {
      "Metric": "Audio streaming support",
      "WebSocket (Ally choice)": "Yes, binary PCM audio both directions [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
      "REST polling": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/HTTP]",
      "SSE": "Server -> client only [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
      "gRPC streaming": "Yes [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
      "LiveKit / WebRTC": "Yes [source: https://docs.livekit.io/intro/basics/connect/]",
      "MQTT": "Message-based audio possible but not browser-native [source: https://mqtt.org/]",
      "Socket.IO": "Possible over WS abstraction [source: https://socket.io/docs/v4/]",
      "HTTP/3 + QUIC streams": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Glossary/QUIC]",
      "Pipecat": "N/A (not publicly available) [source: https://github.com/pipecat-ai/pipecat]",
      "Daily.co": "Yes [source: https://docs.daily.co/]",
      "Agora RTE": "Yes [source: https://docs.agora.io/en/real-time-engagement]"
    },
    {
      "Metric": "Barge-in support",
      "WebSocket (Ally choice)": "Yes, response.cancel + reconnect logic [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
      "REST polling": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/HTTP]",
      "SSE": "No client->server interruption channel [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
      "gRPC streaming": "Possible with bidi stream [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
      "LiveKit / WebRTC": "Yes [source: https://docs.livekit.io/intro/basics/connect/]",
      "MQTT": "N/A (not publicly available) [source: https://mqtt.org/]",
      "Socket.IO": "N/A (not publicly available) [source: https://socket.io/docs/v4/]",
      "HTTP/3 + QUIC streams": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Glossary/QUIC]",
      "Pipecat": "N/A (not publicly available) [source: https://github.com/pipecat-ai/pipecat]",
      "Daily.co": "Yes [source: https://docs.daily.co/]",
      "Agora RTE": "Yes [source: https://docs.agora.io/en/real-time-engagement]"
    },
    {
      "Metric": "Binary frame support",
      "WebSocket (Ally choice)": "Yes [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
      "REST polling": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/HTTP]",
      "SSE": "No [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
      "gRPC streaming": "Yes [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
      "LiveKit / WebRTC": "Yes [source: https://docs.livekit.io/intro/basics/connect/]",
      "MQTT": "Yes [source: https://mqtt.org/]",
      "Socket.IO": "Yes [source: https://socket.io/docs/v4/]",
      "HTTP/3 + QUIC streams": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Glossary/QUIC]",
      "Pipecat": "N/A (not publicly available) [source: https://github.com/pipecat-ai/pipecat]",
      "Daily.co": "N/A (not publicly available) [source: https://docs.daily.co/]",
      "Agora RTE": "N/A (not publicly available) [source: https://docs.agora.io/en/real-time-engagement]"
    },
    {
      "Metric": "Reconnect strategy",
      "WebSocket (Ally choice)": "Custom reconnect in browser + backend session reconnect [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
      "REST polling": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/HTTP]",
      "SSE": "Browser reconnect semantics via EventSource [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
      "gRPC streaming": "N/A (not publicly available) [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
      "LiveKit / WebRTC": "Automatic reconnection documented [source: https://docs.livekit.io/intro/basics/connect/]",
      "MQTT": "N/A (not publicly available) [source: https://mqtt.org/]",
      "Socket.IO": "Built-in reconnection support [source: https://socket.io/docs/v4/]",
      "HTTP/3 + QUIC streams": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Glossary/QUIC]",
      "Pipecat": "N/A (not publicly available) [source: https://github.com/pipecat-ai/pipecat]",
      "Daily.co": "N/A (not publicly available) [source: https://docs.daily.co/]",
      "Agora RTE": "N/A (not publicly available) [source: https://docs.agora.io/en/real-time-engagement]"
    },
    {
      "Metric": "Browser-native support",
      "WebSocket (Ally choice)": "Yes [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
      "REST polling": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/HTTP]",
      "SSE": "Yes [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
      "gRPC streaming": "Not browser-native without extra layer [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
      "LiveKit / WebRTC": "Requires SDK/media stack [source: https://docs.livekit.io/intro/basics/connect/]",
      "MQTT": "N/A (not publicly available) [source: https://mqtt.org/]",
      "Socket.IO": "N/A (not publicly available) [source: https://socket.io/docs/v4/]",
      "HTTP/3 + QUIC streams": "No simple browser API equivalent [source: https://developer.mozilla.org/en-US/docs/Glossary/QUIC]",
      "Pipecat": "N/A (not publicly available) [source: https://github.com/pipecat-ai/pipecat]",
      "Daily.co": "N/A (not publicly available) [source: https://docs.daily.co/]",
      "Agora RTE": "N/A (not publicly available) [source: https://docs.agora.io/en/real-time-engagement]"
    },
    {
      "Metric": "TLS / encryption requirement",
      "WebSocket (Ally choice)": "WSS in production [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
      "REST polling": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/HTTP]",
      "SSE": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
      "gRPC streaming": "N/A (not publicly available) [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
      "LiveKit / WebRTC": "Yes, secure transport family [source: https://docs.livekit.io/intro/basics/connect/]",
      "MQTT": "N/A (not publicly available) [source: https://mqtt.org/]",
      "Socket.IO": "N/A (not publicly available) [source: https://socket.io/docs/v4/]",
      "HTTP/3 + QUIC streams": "Yes [source: https://developer.mozilla.org/en-US/docs/Glossary/QUIC]",
      "Pipecat": "N/A (not publicly available) [source: https://github.com/pipecat-ai/pipecat]",
      "Daily.co": "N/A (not publicly available) [source: https://docs.daily.co/]",
      "Agora RTE": "N/A (not publicly available) [source: https://docs.agora.io/en/real-time-engagement]"
    },
    {
      "Metric": "Backpressure / flow control",
      "WebSocket (Ally choice)": "MDN notes no built-in backpressure [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
      "REST polling": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/HTTP]",
      "SSE": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
      "gRPC streaming": "Streaming flow control supported [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
      "LiveKit / WebRTC": "N/A (not publicly available) [source: https://docs.livekit.io/intro/basics/connect/]",
      "MQTT": "QoS options [source: https://mqtt.org/]",
      "Socket.IO": "Library-level buffering [source: https://socket.io/docs/v4/]",
      "HTTP/3 + QUIC streams": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Glossary/QUIC]",
      "Pipecat": "N/A (not publicly available) [source: https://github.com/pipecat-ai/pipecat]",
      "Daily.co": "N/A (not publicly available) [source: https://docs.daily.co/]",
      "Agora RTE": "N/A (not publicly available) [source: https://docs.agora.io/en/real-time-engagement]"
    },
    {
      "Metric": "Firewall / NAT robustness",
      "WebSocket (Ally choice)": "Works broadly through standard web ports [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
      "REST polling": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/HTTP]",
      "SSE": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
      "gRPC streaming": "N/A (not publicly available) [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
      "LiveKit / WebRTC": "Strong NAT traversal story [source: https://docs.livekit.io/intro/basics/connect/]",
      "MQTT": "N/A (not publicly available) [source: https://mqtt.org/]",
      "Socket.IO": "N/A (not publicly available) [source: https://socket.io/docs/v4/]",
      "HTTP/3 + QUIC streams": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Glossary/QUIC]",
      "Pipecat": "N/A (not publicly available) [source: https://github.com/pipecat-ai/pipecat]",
      "Daily.co": "N/A (not publicly available) [source: https://docs.daily.co/]",
      "Agora RTE": "Designed for realtime traversal [source: https://docs.agora.io/en/real-time-engagement]"
    },
    {
      "Metric": "Typical mobile-network latency note",
      "WebSocket (Ally choice)": "Measured local connection overhead near zero after reuse; end-to-end depends on model turn [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
      "REST polling": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/HTTP]",
      "SSE": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
      "gRPC streaming": "N/A (not publicly available) [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
      "LiveKit / WebRTC": "Designed for realtime media delivery [source: https://docs.livekit.io/intro/basics/connect/]",
      "MQTT": "N/A (not publicly available) [source: https://mqtt.org/]",
      "Socket.IO": "N/A (not publicly available) [source: https://socket.io/docs/v4/]",
      "HTTP/3 + QUIC streams": "Built for low-latency transport [source: https://developer.mozilla.org/en-US/docs/Glossary/QUIC]",
      "Pipecat": "N/A (not publicly available) [source: https://github.com/pipecat-ai/pipecat]",
      "Daily.co": "N/A (not publicly available) [source: https://docs.daily.co/]",
      "Agora RTE": "N/A (not publicly available) [source: https://docs.agora.io/en/real-time-engagement]"
    },
    {
      "Metric": "Implementation complexity",
      "WebSocket (Ally choice)": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
      "REST polling": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/HTTP]",
      "SSE": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
      "gRPC streaming": "N/A (not publicly available) [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
      "LiveKit / WebRTC": "N/A (not publicly available) [source: https://docs.livekit.io/intro/basics/connect/]",
      "MQTT": "N/A (not publicly available) [source: https://mqtt.org/]",
      "Socket.IO": "N/A (not publicly available) [source: https://socket.io/docs/v4/]",
      "HTTP/3 + QUIC streams": "N/A (not publicly available) [source: https://developer.mozilla.org/en-US/docs/Glossary/QUIC]",
      "Pipecat": "N/A (not publicly available) [source: https://github.com/pipecat-ai/pipecat]",
      "Daily.co": "N/A (not publicly available) [source: https://docs.daily.co/]",
      "Agora RTE": "N/A (not publicly available) [source: https://docs.agora.io/en/real-time-engagement]"
    }
  ],
  "criteria": [
    "duplex audio",
    "interruptions",
    "browser fit",
    "low complexity"
  ],
  "fit_matrix": {
    "WebSocket (Ally choice)": [
      1,
      1,
      1,
      1
    ],
    "REST polling": [
      0,
      0,
      1,
      0.2
    ],
    "SSE": [
      0,
      0,
      1,
      0.6
    ],
    "gRPC streaming": [
      1,
      1,
      0.3,
      0.2
    ],
    "LiveKit / WebRTC": [
      1,
      1,
      0.8,
      0.3
    ],
    "MQTT": [
      0.5,
      0.3,
      0.2,
      0.4
    ],
    "Socket.IO": [
      1,
      0.8,
      1,
      0.5
    ],
    "HTTP/3 + QUIC streams": [
      0.5,
      0.4,
      0.1,
      0.1
    ],
    "Pipecat": [
      1,
      1,
      0.2,
      0.3
    ],
    "Daily.co": [
      1,
      1,
      0.7,
      0.3
    ],
    "Agora RTE": [
      1,
      1,
      0.7,
      0.3
    ]
  }
}


In [ ]:
import pandas as pd
df = pd.DataFrame(data["comparison_rows"])
display(df)


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir=project_root/'docs'/'comparisons'/'charts'
charts_dir.mkdir(parents=True,exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
 colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label==list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
 ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
labels=list(data['fit_matrix'].keys())
values=[sum(v) for v in data['fit_matrix'].values()]
fig,ax=plt.subplots(figsize=(12,5)); style(ax,"Ally Vision v2 — Category 8 - WebSocket Architecture Comparison Overall Fit Score")
ax.bar(labels,values,color=colors); ax.set_ylabel('Derived fit score',color='white'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.savefig(charts_dir / 'category8_websocket_architecture_comparison_chart1.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir=project_root/'docs'/'comparisons'/'charts'
charts_dir.mkdir(parents=True,exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
 colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label==list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
 ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
criteria=data['criteria']; selected=["WebSocket (Ally choice)", "LiveKit / WebRTC", "Daily.co", "Agora RTE", "Socket.IO"]; x=np.arange(len(criteria)); width=0.8/len(selected)
fig,ax=plt.subplots(figsize=(12,5)); style(ax,"Ally Vision v2 — Category 8 - WebSocket Architecture Comparison Top-5 Criteria View")
for idx,label in enumerate(selected):
 vals=data['fit_matrix'][label]; color=ALLY if label==selected[0] else (DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else COMP); ax.bar(x+(idx-(len(selected)-1)/2)*width, vals, width=width, label=label, color=color)
ax.set_xticks(x); ax.set_xticklabels(criteria, rotation=20, ha='right', color='white'); ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white'); plt.tight_layout(); plt.savefig(charts_dir / 'category8_websocket_architecture_comparison_chart2.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir=project_root/'docs'/'comparisons'/'charts'
charts_dir.mkdir(parents=True,exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
 colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label==list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
 ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
criteria=data['criteria']; selected=["WebSocket (Ally choice)", "LiveKit / WebRTC", "Daily.co", "Agora RTE", "Socket.IO"]; mat=np.array([data['fit_matrix'][k] for k in selected])
fig,ax=plt.subplots(figsize=(10,5)); ax.set_facecolor(BG); ax.figure.set_facecolor(BG); im=ax.imshow(mat,cmap='Blues',aspect='auto'); ax.set_title('Ally Vision v2 — Category 8 - WebSocket Architecture Comparison Trade-off Heatmap',color='white'); ax.set_xticks(np.arange(len(criteria))); ax.set_xticklabels(criteria, rotation=20, ha='right', color='white'); ax.set_yticks(np.arange(len(selected))); ax.set_yticklabels(selected,color='white')
for i in range(mat.shape[0]):
 for j in range(mat.shape[1]): ax.text(j,i,f"{mat[i,j]:.0f}",ha='center',va='center',color='white')
plt.colorbar(im); plt.tight_layout(); plt.savefig(charts_dir / 'category8_websocket_architecture_comparison_chart3.png',dpi=150,bbox_inches='tight'); plt.show()


## Data Sources

| # | Source Name | URL | Value extracted |
|---|-------------|-----|-----------------|
| 1 | MDN WebSocket | https://developer.mozilla.org/en-US/docs/Web/API/WebSocket | browser support and backpressure note |
| 2 | MDN SSE | https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events | one-way stream and reconnection limits |
| 3 | gRPC core concepts | https://grpc.io/docs/what-is-grpc/core-concepts/ | bidirectional streaming support |
| 4 | LiveKit connect docs | https://docs.livekit.io/intro/basics/connect/ | room sessions and reconnection |
| 5 | Project realtime client | https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py | 120 min session and reconnect threshold |


## CONCLUSION

Persistent WebSockets remain the best practical transport choice for Ally Vision v2. They support duplex audio, interruption behavior, browser simplicity, and session reuse without forcing the project into a heavier media framework than it currently needs.

→ Chosen for Ally Vision v2 ✅
